# T2. KNN y SVM
**Extraccion del conocimiento en Bases de datos**  
Leobardo Daniel Bertadillo Villalobos - 9C

Clasificacion de siluetas de vehiculos (dataset Statlog) comparando KNN y SVM.
Las metricas de evaluacion se calculan a mano a partir de la matriz de confusion.

## 1. Importar librerias

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

sns.set_style("whitegrid")

## 2. Cargar y explorar los datos

In [ ]:
df = pd.read_csv("vehicle.csv")
df.head()

In [ ]:
# Tamano del conjunto y revision de clases
print("Instancias:", df.shape[0], "| Columnas:", df.shape[1])
print("Valores nulos:", df.isnull().sum().sum())
df["target"].value_counts().sort_index()

In [ ]:
# Distribucion de clases
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="target")
plt.title("Distribucion de clases")
plt.xlabel("Clase")
plt.ylabel("Instancias")
plt.tight_layout()
plt.show()

## 3. Preparacion de los datos

Separamos las caracteristicas de la clase, dividimos en entrenamiento y prueba
y estandarizamos las variables (importante porque KNN y SVM dependen de distancias).

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]
etiquetas = sorted(y.unique())

# 75% entrenamiento y 25% prueba, manteniendo la proporcion de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print("Entrenamiento:", X_train.shape[0], "| Prueba:", X_test.shape[0])

In [ ]:
# El escalador se ajusta solo con los datos de entrenamiento
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 4. Funciones para calcular las metricas a mano

En lugar de usar las funciones de metricas de la libreria, las calculamos
directamente. Primero armamos la matriz de confusion contando, para cada par,
cuantas veces una clase real se predijo como otra.

Para varias clases cada una se evalua una contra el resto (one-vs-rest):

- TP: instancias de la clase clasificadas correctamente.
- FN: instancias de la clase asignadas a otra clase.
- FP: instancias de otras clases asignadas a esta clase.
- TN: el resto de las instancias.

Con esos valores se obtienen las formulas vistas en clase:

- Exactitud (accuracy) = aciertos totales / total de instancias
- Precision = TP / (TP + FP)
- Sensibilidad (recall) = TP / (TP + FN)
- Especificidad = TN / (TN + FP)

In [ ]:
def matriz_confusion(y_real, y_pred, etiquetas):
    # Cuenta cuantas veces la clase real se predijo como cada clase
    indice = {c: i for i, c in enumerate(etiquetas)}
    m = np.zeros((len(etiquetas), len(etiquetas)), dtype=int)
    for real, pred in zip(y_real, y_pred):
        m[indice[real], indice[pred]] += 1
    return m


def accuracy_global(cm):
    # Aciertos (diagonal) entre el total
    return np.trace(cm) / cm.sum()


def metricas_por_clase(cm, etiquetas):
    total = cm.sum()
    filas = []
    for i, c in enumerate(etiquetas):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP   # esta clase clasificada en otra
        FP = cm[:, i].sum() - TP   # otras clases asignadas a esta
        TN = total - TP - FN - FP
        precision = TP / (TP + FP)
        sensibilidad = TP / (TP + FN)
        especificidad = TN / (TN + FP)
        f1 = 2 * precision * sensibilidad / (precision + sensibilidad)
        filas.append([c, TP, FN, FP, TN, precision, sensibilidad, especificidad, f1])
    columnas = ["Clase", "TP", "FN", "FP", "TN",
                "Precision", "Sensibilidad", "Especificidad", "F1"]
    return pd.DataFrame(filas, columns=columnas)


def graficar_matriz(cm, etiquetas, titulo):
    plt.figure(figsize=(5, 4.2))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=etiquetas, yticklabels=etiquetas)
    plt.title(titulo)
    plt.xlabel("Prediccion")
    plt.ylabel("Valor real")
    plt.tight_layout()
    plt.show()

## 5. KNN

Probamos varios valores de k con validacion cruzada y nos quedamos con el mejor.

In [ ]:
rango_k = range(1, 31)
promedios = []

for k in rango_k:
    modelo = KNeighborsClassifier(n_neighbors=k)
    puntajes = cross_val_score(modelo, X_train, y_train, cv=5, scoring="accuracy")
    promedios.append(puntajes.mean())

mejor_k = list(rango_k)[int(np.argmax(promedios))]
print("Mejor k:", mejor_k, "| Accuracy CV:", round(max(promedios), 4))

In [ ]:
# Grafica de accuracy contra k
plt.figure(figsize=(7, 4))
plt.plot(list(rango_k), promedios, marker="o", color="#003366")
plt.axvline(mejor_k, color="#cc6600", linestyle="--", label=f"k optimo = {mejor_k}")
plt.xlabel("Numero de vecinos (k)")
plt.ylabel("Accuracy promedio (validacion cruzada)")
plt.title("Seleccion de k para KNN")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Entrenamos el KNN final y medimos tiempos
knn = KNeighborsClassifier(n_neighbors=mejor_k)

inicio = time.perf_counter()
knn.fit(X_train, y_train)
t_entrena_knn = (time.perf_counter() - inicio) * 1000

inicio = time.perf_counter()
y_pred_knn = knn.predict(X_test)
t_predice_knn = (time.perf_counter() - inicio) * 1000

## 6. SVM

Buscamos el mejor kernel y sus parametros con una rejilla de busqueda.

In [ ]:
rejilla = [
    {"kernel": ["linear"], "C": [0.1, 1, 10, 100]},
    {"kernel": ["rbf"], "C": [0.1, 1, 10, 100], "gamma": ["scale", 0.01, 0.1, 1]},
    {"kernel": ["poly"], "C": [0.1, 1, 10], "degree": [2, 3], "gamma": ["scale"]},
]

busqueda = GridSearchCV(SVC(), rejilla, cv=5, scoring="accuracy", n_jobs=-1)
busqueda.fit(X_train, y_train)

print("Mejor configuracion:", busqueda.best_params_)
print("Accuracy CV:", round(busqueda.best_score_, 4))

In [ ]:
# Entrenamos el SVM final y medimos tiempos
svm = SVC(**busqueda.best_params_)

inicio = time.perf_counter()
svm.fit(X_train, y_train)
t_entrena_svm = (time.perf_counter() - inicio) * 1000

inicio = time.perf_counter()
y_pred_svm = svm.predict(X_test)
t_predice_svm = (time.perf_counter() - inicio) * 1000

## 7. Matrices de confusion

Se construyen a mano con la funcion definida arriba.

In [ ]:
cm_knn = matriz_confusion(y_test, y_pred_knn, etiquetas)
print("Matriz de confusion KNN:")
print(cm_knn)
graficar_matriz(cm_knn, etiquetas, "Matriz de confusion - KNN")

In [ ]:
cm_svm = matriz_confusion(y_test, y_pred_svm, etiquetas)
print("Matriz de confusion SVM:")
print(cm_svm)
graficar_matriz(cm_svm, etiquetas, "Matriz de confusion - SVM")

## 8. Calculo manual de las metricas

Con las matrices de confusion calculamos precision, sensibilidad,
especificidad y F1 por clase, ademas de la exactitud global de cada modelo.

In [ ]:
# Metricas por clase de KNN
tabla_knn = metricas_por_clase(cm_knn, etiquetas)
tabla_knn.round(4)

In [ ]:
# Metricas por clase de SVM
tabla_svm = metricas_por_clase(cm_svm, etiquetas)
tabla_svm.round(4)

In [ ]:
# Exactitud global de cada modelo
acc_knn = accuracy_global(cm_knn)
acc_svm = accuracy_global(cm_svm)
print("Accuracy KNN:", round(acc_knn, 4))
print("Accuracy SVM:", round(acc_svm, 4))

## 9. Comparacion de los modelos

El Macro F1 es el promedio del F1 de las cuatro clases.

In [ ]:
f1_knn = tabla_knn["F1"].mean()   # Macro F1
f1_svm = tabla_svm["F1"].mean()

comparativa = pd.DataFrame({
    "Modelo": [f"KNN (k={mejor_k})", "SVM"],
    "Accuracy": [acc_knn, acc_svm],
    "Macro F1": [f1_knn, f1_svm],
    "Entrenamiento (ms)": [t_entrena_knn, t_entrena_svm],
    "Prediccion (ms)": [t_predice_knn, t_predice_svm],
})
comparativa.round(4)

## 10. Conclusion

El SVM con kernel RBF obtiene la mejor exactitud y el mejor Macro F1.
Los errores se concentran entre las dos clases de automoviles, cuyas
siluetas son geometricamente muy parecidas.